<a href="https://colab.research.google.com/github/chidiview-ui/shiny-telegram/blob/main/Context_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install crewai
from crewai import Crew, Agent, Task
from crewai.flow.flow import Flow, listen, start

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.4/814.4 

In [ ]:
class ContextEngineeringFlow(Flow):
    @start
    def process_query(self):           # Save user query to memory
        self.memory_layer.save_user_message(self.state.query)
        return(self.state.query)

    @listen(process_query)            # Gather context from all sources
    async def gather_query(self):
        context_crew = Crew(
            agents=[rag_agent, memory_agent, web_serach_agent, arxiv_api_agent],
            tasks=[rag_task, memory_task, web_serach_task, arxiv_api_task]
        )
        results=await context_crew.kickoff_async()
        return results

    @listen(gather_query)  # Filter out irrelevant context
    def evaluate_context_relevance(self, flow_state):
        evaluation_result = evaluation_crew.kickoff()
        filtered_context = filter_context(evaluation_result)
        return filtered_context

    @listen(evaluate_context_relevance)
    def synthesize_final_response(self, flow_state):
        synthesis_result = synthesis_crew.kickoff()
        final_response = synthesis_results.tasks_outputs[0].raw

        # save assitant response to memory
        self.memory_layer.save_assistant_message(final_response)
        return final_response

In [ ]:
!pip install tensorlake

In [ ]:
import os
!pip install tensorlake

from typing import List
from tensorlake.documentai import DocumentAI, ChunkingStrategy, ParsingOptions, TableOutputMode, StructuredExtractionOptions
print(ChunkingStrategy)
from pydantic import BaseModel, Field


# Assuming TENSOR_LAKE_API_KEY is an environment variable or defined elsewhere
TENSOR_LAKE_API_KEY = os.getenv("Agentic") # Placeholder

class Section(BaseModel):
  heading: str = Field(description = "The section heading")
  summary: str = Field (description = "The summary of the section content")

class ResearchPaper(BaseModel):
    title: str = Field(description = "The title of the research paper")
    authors: List[str] = Field(description = "List of paper authors")
    abstract: str = Field(description = "The Paper's Abstract")
    sections: List[Section] = Field (description = "Sections with heading and summaries")

from google.colab import userdata
Agentic = userdata.get('Agentic')
doc_ai = DocumentAI(api_key = Agentic)

# Placeholder for a valid PDF path. You need to replace this with an actual file path.
# For example, if you have a PDF named 'my_research_paper.pdf' in the same directory as your notebook:
# file_id = doc_ai.upload(path="./my_research_paper.pdf")
# For demonstration, I'll use a dummy path.
file_id = doc_ai.upload(path= "/content/drive/MyDrive/my_research_paper.pdf") # Changed to a PDF placeholder

parsing_options = ParsingOptions(
    chunking_strategy=ChunkingStrategy.SECTION, # Changed AUTO to SECTION
    table_output_mode=TableOutputMode.NONE
)

research_paper_extraction = StructuredExtractionOptions(
    schema_name = "research_paper",
    json_schema = ResearchPaper.model_json_schema(), # Use .model_json_schema() to get the Pydantic schema
    provide_citations = True
)

parse_id = doc_ai.parse(
    file=file_id,
    parsing_options=parsing_options,
    structured_extraction_options=research_paper_extraction
)

result=doc_ai.wait_for_completion(parse_id)

rag_chunks = [chunk.content for chunk in result.chunks] # Corrected 'results' to 'result'
extracted_data=result.structured_data

<enum 'ChunkingStrategy'>


FileNotFoundError: File not found: /content/drive/MyDrive/my_research_paper.pdf

In [ ]:
print(dir(ChunkingStrategy))

['FRAGMENT', 'NONE', 'PAGE', 'SECTION', '__add__', '__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__members__', '__mod__', '__module__', '__mul__', '__name__', '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__rmod__', '__rmul__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'capitalize', 'casefold', 'center', 'count', 'encode', 'endswith', 'expandtabs', 'find', 'format', 'format_map', 'index', 'isalnum', 'isalpha', 'isascii', 'isdecimal', 'isdigit', 'isidentifier', 'islower', 'isnumeric', 'isprintable', 'isspace', 'istitle', 'isupper', 'join', 'ljust', 'lower', 'lstrip', 'maketrans', 'partition', 'removeprefix', 'removesuffix', 'replace', 'rfind', 'rindex', 'rjust', 'rpartition', 'rsplit', 'rstrip', 'split', 'splitlines'

In [ ]:
# This cell is no longer needed as its content has been merged into cell dBdsSJwSZoWO.

In [ ]:
# This cell is no longer needed as its content has been merged into cell dBdsSJwSZoWO.